In [270]:
import pandas as pd
import math
import numpy as np
from enum import Enum

In [271]:
#Import set of generated answers for this set of users
df_data = pd.read_csv("SADD_Options_Selected_4users.csv")
print(df_data.sort_values(by="Group") )


   User_ID  Group  RBAC  ABAC  RBAC_ABAC
0        1      1     5     2          1
1        2      1     3     4          5
3        4      2     5     2          3
2        3      3     3     2          2


In [272]:
class PreferenceIndication(Enum) :
    RANKING = 1
    RATING = 2
    YesNo = 3
class TeamProfiles(Enum) :
    EXPERT = 1
    ADVANCED = 2
    BEGINNER = 3

class Algorithm :
    def __init__(self,name,algorithm) :
        self.name = name 
        self.algorithm = algorithm
        
    def compute(self,df) :
        pass
        

class DecisionRule :
    def __init__(self,name,rule,teamprofiles : list[TeamProfiles],max_accepted_value):
        self.name = name
        self.rule = rule
        self.fortp = teamprofiles
        self.max_accepted_value = max_accepted_value
    
    # Filters the dataframe we give with the team profiles defined before
    def get_df_filtered(self,df) :
        strtp = "Group == " + str(self.fortp[0].value)
        
        for tp in range(1,len(self.fortp)) :
            strtp += "| Group == " + str(self.fortp[tp].value)
        return strtp
            
    def str_tp(self) :
        strtp = ""
        for tp in self.fortp :
            strtp += tp.name + " "
        return strtp
        
    def compute(self,df) :
        print("=======================")
        print("Applying " + self.name + " for team profiles: "+ self.str_tp())
        # We filter the dataframe to retrieve only the team profiles specified in the decision rule
        filtertp = self.get_df_filtered(df)
        df_filtered = df.query(filtertp)
        # We retrieve the alternatives column names
        alternatives = df_filtered.columns.tolist()[2:]
        
        # Retrieve the sum of ratings for each alternative
        sum_results = dict()
        for alt in alternatives :
            sum_results.update({alt : df_filtered[alt].sum()})
        total_votes = self.max_accepted_value * len(df_filtered.index)
        
        result = []
        # Go through the alternatives sum and check if the rule given complies
        for key,val in sum_results.items() :
            # If it complies, add the alternative to the result list
            print(f"[{key}]: {val}/{total_votes}")
            if self.rule(val,total_votes) :
                result.append(key)
        # Print the alternatives found if there is multiple or none or the only suggested one
        if len(result) > 1 :
            print("Multiple alternatives matches were found: " + str(result))
        elif len(result) == 0:
            print("No alternatives were found")
        else :
            print("Suggested alternative: " + str(result[0]))
        print("=======================")

class Strategy :
    def __init__(self,name: str,preference_indication: PreferenceIndication,strategy_sequence: list[Algorithm,DecisionRule]) :
        self.name = name
        self.preference_indication = preference_indication
        self.strategy_sequence = strategy_sequence
        
    
    def add_decision_rule(self,decision_rule) :
        self.strategy_sequence.append(decision_rule)
        
    def add_algorithm(self,algorithm) :
        self.strategy_sequence.append(algorithm)
        
    def compute_strat(self,df) :
        print(f"Applying Strategy {self.name} with a preference indication {self.preference_indication}:")
        for elt in self.strategy_sequence :
            elt.compute(df)
    

        

In [ ]:
# With this decision rule we 
majority = DecisionRule("Majority",lambda nb_votes,total_votes : nb_votes > total_votes*50/100 , [TeamProfiles.EXPERT,TeamProfiles.ADVANCED,TeamProfiles.BEGINNER],5)
unanimity_exp_adv = DecisionRule("Unanimity Exp_Adv",lambda nb_votes,total_votes : nb_votes == total_votes, [TeamProfiles.EXPERT,TeamProfiles.ADVANCED],5)
super_majority_exp_adv = DecisionRule("Super Majority Exp_Adv",lambda nb_votes,total_votes : nb_votes > total_votes*65/100, [TeamProfiles.EXPERT,TeamProfiles.ADVANCED],5)

unanimity = DecisionRule("Unanimity",lambda nb_votes,total_votes : nb_votes == total_votes, [TeamProfiles.EXPERT,TeamProfiles.ADVANCED,TeamProfiles.BEGINNER],5)

strat = Strategy("Voting",PreferenceIndication.RATING,[unanimity])
strat_2 = Strategy("StratAuthorizationActiveComponent",PreferenceIndication.RATING,[unanimity_exp_adv,super_majority_exp_adv])

design_time_strategies = [strat,strat_2]

for strat in design_time_strategies :
    strat.compute_strat(df_data)

   

Applying Strategy Voting with a preference indication PreferenceIndication.RATING:
Applying Unanimity for team profiles: EXPERT ADVANCED BEGINNER 
[RBAC]: 16/20
[ABAC]: 10/20
[RBAC_ABAC]: 11/20
No alternatives were found
Applying Strategy StratAuthorizationActiveComponent with a preference indication PreferenceIndication.RATING:
Applying Unanimity Exp_Adv for team profiles: EXPERT ADVANCED 
[RBAC]: 13/15
[ABAC]: 8/15
[RBAC_ABAC]: 9/15
No alternatives were found
Applying Super Majority Exp_Adv for team profiles: EXPERT ADVANCED 
[RBAC]: 13/15
[ABAC]: 8/15
[RBAC_ABAC]: 9/15
Suggested alternative: RBAC
